In [4]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# 1) USER INPUTS
# =========================
DTA_PATH = r"G:\My Drive\uc3m PhD\05 Analysis\01 Main\01 Stata\01 Main\04 Matching quality\out\02. SMDs.dta"
OUT_DIR =  r"G:\My Drive\uc3m PhD\05 Analysis\01 Main\01 Stata\01 Main\04 Matching quality\out"

# =========================
# 2) LOAD DATA
# =========================
df = pd.read_stata(DTA_PATH)

if "covariate" not in df.columns and "cov" in df.columns:
    df = df.rename(columns={"cov": "covariate"})

required_cols = ["acq_type", "bl_tt", "lambda_val", "covariate", "smd"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[required_cols].copy()
df["lambda_val"] = pd.to_numeric(df["lambda_val"], errors="coerce")
df["smd"] = pd.to_numeric(df["smd"], errors="coerce")
df["abs_smd"] = df["smd"].abs()

# =========================
# 3) COVARIATE GROUPS
# =========================
main_covs = ["cit_m4", "cit_m3", "cit_m2", "cit_m1", "num_claim", "age"]

def is_pc(x: str) -> bool:
    return bool(re.fullmatch(r"pc\d+", str(x).lower()))

def pc_num(x: str) -> int:
    m = re.fullmatch(r"pc(\d+)", str(x).lower())
    return int(m.group(1)) if m else 10**9

pc_covs = sorted([c for c in df["covariate"].dropna().unique() if is_pc(c)], key=pc_num)

# =========================
# 4) LAMBDA ORDER / LABELS
# =========================
lambda_vals = sorted(v for v in df["lambda_val"].dropna().unique() if v != 99)
lambda_order = ([99] if 99 in df["lambda_val"].values else []) + lambda_vals

def format_lambda(v):
    if pd.isna(v):
        return "NA"
    if v == 99:
        return "Before matching"
    if float(v).is_integer():
        return f"λ = {int(v)}"
    return f"λ = {v}"

lambda_labels = {v: format_lambda(v) for v in lambda_order}

# colors / markers
cmap = plt.cm.get_cmap("tab10", max(len(lambda_order), 3))
lambda_colors = {v: cmap(i) for i, v in enumerate(lambda_order)}
lambda_markers = {v: "o" for v in lambda_order}
if 99 in lambda_order:
    lambda_colors[99] = "black"
    lambda_markers[99] = "D"

# small vertical offsets so all lambdas appear on same covariate row
if len(lambda_order) == 1:
    offsets = {lambda_order[0]: 0.0}
else:
    spread = 0.28
    vals = np.linspace(-spread, spread, len(lambda_order))
    offsets = {lam: off for lam, off in zip(lambda_order, vals)}

# =========================
# 5) LOVE PLOT FUNCTION
# =========================
def make_love_plot(data, cov_order, title, save_path, figsize=(9, 5), x_thresholds=(0.1, 0.2)):
    if data.empty:
        print(f"Skipping empty plot: {title}")
        return

    cov_order = [c for c in cov_order if c in set(data["covariate"])]
    if not cov_order:
        print(f"Skipping plot with no covariates present: {title}")
        return

    y_base = {cov: i for i, cov in enumerate(cov_order)}
    plot_df = data[data["covariate"].isin(cov_order)].copy()
    plot_df["y"] = plot_df["covariate"].map(y_base)

    if len(cov_order) > 12:
        figsize = (10, max(6, 0.38 * len(cov_order)))

    fig, ax = plt.subplots(figsize=figsize)

    # thresholds
    for thr in x_thresholds:
        ax.axvline(thr, linestyle="--", linewidth=1, color="gray", alpha=0.8)
    ax.axvline(0, linewidth=1, color="black")

    # plot one scatter layer per lambda, offset vertically within the same covariate row
    for lam in lambda_order:
        sub = plot_df[plot_df["lambda_val"] == lam].copy()
        if sub.empty:
            continue

        sub["y_plot"] = sub["y"] + offsets[lam]

        ax.scatter(
            sub["abs_smd"],
            sub["y_plot"],
            s=70 if lam != 99 else 95,
            marker=lambda_markers[lam],
            color=lambda_colors[lam],
            edgecolor="black" if lam == 99 else "white",
            linewidth=0.9,
            alpha=0.95,
            label=lambda_labels[lam],
            zorder=4 if lam == 99 else 3
        )

    ax.set_yticks(range(len(cov_order)))
    ax.set_yticklabels(cov_order)
    ax.invert_yaxis()
    ax.set_xlabel("|SMD|")
    ax.set_title(title)

    # unique legend entries in desired order
    handles, labels = ax.get_legend_handles_labels()
    seen = set()
    handles2, labels2 = [], []
    for h, l in zip(handles, labels):
        if l not in seen:
            handles2.append(h)
            labels2.append(l)
            seen.add(l)
    ax.legend(handles2, labels2, title="Matching status / lambda", loc="best", frameon=True)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

# =========================
# 6) GENERATE PLOTS
# =========================
groups = (
    df[["acq_type", "bl_tt"]]
    .drop_duplicates()
    .sort_values(["acq_type", "bl_tt"])
    .itertuples(index=False, name=None)
)

for acq_type, bl_tt in groups:
    sub = df[(df["acq_type"] == acq_type) & (df["bl_tt"] == bl_tt)].copy()

    safe_acq = re.sub(r"[^A-Za-z0-9_\-]+", "_", str(acq_type))
    safe_bl = re.sub(r"[^A-Za-z0-9_\-]+", "_", str(bl_tt))

    # main covariates plot
    sub_main = sub[sub["covariate"].isin(main_covs)].copy()
    out_main = os.path.join(OUT_DIR, f"love_main__acq_{safe_acq}__bltt_{safe_bl}.png")
    make_love_plot(
        data=sub_main,
        cov_order=main_covs,
        title=f"Love plot: main covariates\nacq_type={acq_type}, bl_tt={bl_tt}",
        save_path=out_main,
        figsize=(9, 4.8)
    )

    # PCs plot
    sub_pc = sub[sub["covariate"].isin(pc_covs)].copy()
    out_pc = os.path.join(OUT_DIR, f"love_pc__acq_{safe_acq}__bltt_{safe_bl}.png")
    make_love_plot(
        data=sub_pc,
        cov_order=pc_covs,
        title=f"Love plot: principal components\nacq_type={acq_type}, bl_tt={bl_tt}",
        save_path=out_pc,
        figsize=(10, max(7, 0.38 * len(pc_covs)))
    )

print(f"Done. Plots saved to: {OUT_DIR}")

C:\Users\aliwk\AppData\Local\Temp\ipykernel_2528\1308075057.py:63: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap("tab10", max(len(lambda_order), 3))


Done. Plots saved to: G:\My Drive\uc3m PhD\05 Analysis\01 Main\01 Stata\01 Main\04 Matching quality\out


In [6]:
from csdid.att_gt import ATTgt
import pandas as pd
data = pd.read_csv("https://raw.githubusercontent.com/d2cml-ai/csdid/function-aggte/data/mpdta.csv")

In [7]:
out = ATTgt(yname = "lemp",
              gname = "first.treat",
              idname = "countyreal",
              tname = "year",
              xformla = f"lemp~1",
              data = data,
              ).fit(est_method = 'dr')

In [9]:
out.aggte(typec='dynamic');



Overall summary of ATT's based on event-study/dynamic aggregation:
    ATT Std. Error  [95.0%  Conf. Int.]  
-0.0772      0.022 -0.1204      -0.0341 *


Dynamic Effects:
   Event time  Estimate  Std. Error  [95.0% Simult.   Conf. Band   
0          -3    0.0305      0.0158          -0.0005      0.0615   
1          -2   -0.0006      0.0134          -0.0269      0.0258   
2          -1   -0.0245      0.0145          -0.0528      0.0039   
3           0   -0.0199      0.0119          -0.0433      0.0034   
4           1   -0.0510      0.0173          -0.0849     -0.0170  *
5           2   -0.1373      0.0373          -0.2103     -0.0642  *
6           3   -0.1008      0.0359          -0.1712     -0.0305  *
---
Signif. codes: `*' confidence band does not cover 0
Control Group:  Never Treated , 
Anticipation Periods:  0
Estimation Method:  Doubly Robust


